# FeynmanEngine for a BSM Theorist

**Audience**: theorist who wants to plug a custom |M̄|² into the engine — for a new BSM mediator, an SMEFT operator, an extra Higgs, whatever — and have all downstream machinery (σ integration, differential distributions, NLO running-K, hadronic enumeration) work automatically.

**Goal**: register your own amplitude in 4 lines of code and use it like a built-in.

**Prerequisite**: `pip install feynman-engine && feynman setup --skip-lhapdf --skip-openloops` is sufficient (we don't need OpenLoops or precise PDFs for the examples here — but they still work alongside if you have them).

---

**Outline:**
1. The bundled BSM model (`Z'` + scalar dark matter)
2. `register_curated_amplitude` — the public extension API
3. **Example A**: a heavy scalar mediator `S → χ χ̄`
4. **Example B**: an SMEFT-style anomalous TGC contribution to $e^+e^- \to W^+W^-$
5. **Example C**: a leptophobic `Z'` with custom couplings
6. Compute σ + dσ/dcosθ for your registered process — the same path as built-ins

## 1. What ships in the bundled BSM model

The `bsm.mod` model file (used by QGRAF for diagram enumeration) ships with a minimal simplified-DM particle content:
- SM QED-sector fermions: `e±, μ±, A` (photon)
- New mediator: `Zp` (heavy `Z'`, dark photon)
- Dark matter: `chi, chi~` (scalar DM)

This deliberately stops short of full SUSY/SMEFT/2HDM. The intended workflow is: **start from this minimal model, register your own |M̄|² for any process you care about**. The engine treats your formula like any built-in curated amplitude.

In [ ]:
# What's already there?
from fastapi.testclient import TestClient
from feynman_engine.api.app import app
client = TestClient(app)

r = client.get("/api/amplitude/processes").json()
bsm = [p for p in r if p["theory"] == "BSM"]
print(f"Built-in BSM curated entries ({len(bsm)}):")
for p in bsm:
    print(f"  {p['process']}")
print()
print("BSM processes also available via the SymPy symbolic backend (any combination of e±, μ±, γ, Z', χ).")

## 2. The public extension API

```python
from feynman_engine.physics.amplitude import register_curated_amplitude

register_curated_amplitude(
    process    = "a b -> c d",     # canonical process string
    theory     = "BSM",            # theory tag
    msq        = sympy_expression, # spin-averaged |M̄|² in s, t, u and your couplings
    description= "...",
    notes      = "reference, conventions, etc.",
)
```

Conventions (per Peskin & Schroeder eq. 5.79):
- Averaged over initial-state spins / colours
- Summed over final-state spins / polarisations
- **Without** the $1/n!$ identical-particle factor (the σ integrator applies that)
- In terms of the Mandelstam variables `s, t, u` and any couplings/masses you define

Once registered, the engine uses your |M̄|² for: cross-section integration (`/api/amplitude/cross-section`), differential distributions (`/api/amplitude/differential-distribution`), NLO running-coupling rescaling, hadronic enumeration when applicable. **Same chain as built-in curated amplitudes.**

## Example A — Heavy scalar mediator `e⁺ e⁻ → S → χ χ̄`

A toy s-channel scalar singlet that mediates DM production.

Lagrangian:
$$\mathcal{L} \supset -y_e\, S\, \bar{e}e \;-\; y_\chi\, S\, \bar{\chi}\chi$$

Spin-averaged $|\overline{\mathcal{M}}|^2$ for $e^+e^-(p_1,p_2) \to \chi\bar\chi(k_1,k_2)$ via an s-channel scalar of mass $M_S$:

$$|\overline{\mathcal{M}}|^2 = \frac{y_e^2 y_\chi^2}{4} \cdot \frac{(s - 4m_\chi^2)\,s}{(s - M_S^2)^2 + M_S^2 \Gamma_S^2}$$

In [ ]:
import sympy as sp
from feynman_engine.physics.amplitude import register_curated_amplitude

s, t, u = sp.symbols("s t u", real=True)
y_e, y_chi = sp.symbols("y_e y_chi", positive=True)
M_S, Gamma_S, m_chi = sp.symbols("M_S Gamma_S m_chi", positive=True)

# Breit-Wigner s-channel scalar
msq_S = (y_e**2 * y_chi**2 / 4) * (s - 4*m_chi**2) * s / ((s - M_S**2)**2 + M_S**2 * Gamma_S**2)

register_curated_amplitude(
    process="e+ e- -> chi chi~",
    theory="BSM",
    msq=msq_S,
    description="Heavy scalar mediator s-channel: e+e- → S* → χχ̄",
    notes="Lagrangian: -y_e S ē e - y_χ S χ̄ χ.  Replaces the bundled Z'-exchange version.",
    overwrite=True,    # overwrite the built-in Z'-mediated entry for this demo
)
print("Registered: e+ e- -> chi chi~ via heavy scalar mediator S.")
print(f"|M̄|² = {msq_S}")

Now compute σ at √s = 200 GeV with concrete numerical couplings — the engine substitutes them automatically. Pass them via the API's `coupling_overrides` parameter (or the Python `coupling_vals` dict).

In [ ]:
from feynman_engine.amplitudes.cross_section import total_cross_section

r = total_cross_section(
    "e+ e- -> chi chi~", "BSM", sqrt_s=200.0,
    coupling_vals={"y_e": 0.01, "y_chi": 1.0,
                   "M_S": 150.0, "Gamma_S": 1.0, "m_chi": 10.0},
)
print(f"σ(e+e- → χχ̄ via S) at √s=200 GeV = {r['sigma_pb']:.3e} pb")
print(f"  (M_S = 150 GeV is below √s — small Breit-Wigner suppression off-resonance)")

## Example B — SMEFT-style anomalous TGC in $e^+e^- \to W^+W^-$

A common SMEFT scenario: the operator $\mathcal{O}_{WWW} = \epsilon^{abc}W^{a\,\nu}_\mu W^{b\,\rho}_\nu W^{c\,\mu}_\rho$ generates an anomalous WWγ/WWZ vertex. At leading order in $1/\Lambda^2$ the squared amplitude gets a correction:

$$|\overline{\mathcal{M}}|^2_{NP} = \Big|\mathcal{M}_{SM}\Big|^2 \times \Big(1 + 2\, \frac{c_{WWW}}{\Lambda^2}\,s\Big) + \mathcal{O}(\Lambda^{-4})$$

We won't bother computing $\mathcal{M}_{SM}$ from scratch (the engine has it); we just use a toy SMEFT-style enhancement on a simplified Born structure.

In [ ]:
c_WWW, Lambda = sp.symbols("c_WWW Lambda", positive=True)
g_W, m_W = sp.symbols("g_W m_W", positive=True)

# Toy: small-angle WW production with linear SMEFT correction
msq_SM_toy = g_W**4 * (s**2 + 2*s*t + 2*t**2) / (4 * (s - 4*m_W**2)**2)
smeft_correction = 1 + 2 * c_WWW / Lambda**2 * s
msq_SMEFT = msq_SM_toy * smeft_correction

register_curated_amplitude(
    process="e+ e- -> W+ W-",
    theory="BSM",   # register under BSM to avoid clobbering the EW built-in
    msq=msq_SMEFT,
    description="SMEFT O_WWW correction to e+e- → W+W-, leading order in 1/Λ²",
    notes="Toy SM piece × (1 + 2 c_WWW/Λ² × s).  Replace msq_SM_toy with the full HPZ formula for real physics.",
    overwrite=True,
)
print("Registered: e+ e- -> W+ W- with anomalous TGC (BSM tag)")
print(f"|M̄|²_SMEFT = {msq_SMEFT}")

In [ ]:
# Sweep c_WWW/Λ² to see the SMEFT effect
import numpy as np
import matplotlib.pyplot as plt

sqrts = 1000.0  # 1 TeV linear collider
g_W_val = 0.65
m_W_val = 80.4
Lambda_val = 1000.0  # TeV-scale new physics

wilson_coeffs = np.linspace(-2, 2, 9)  # in TeV^-2
sigmas = []
for c in wilson_coeffs:
    r = total_cross_section(
        "e+ e- -> W+ W-", "BSM", sqrt_s=sqrts,
        coupling_vals={"g_W": g_W_val, "m_W": m_W_val,
                       "c_WWW": c, "Lambda": Lambda_val},
    )
    sigmas.append(r["sigma_pb"])

plt.figure(figsize=(7, 4))
plt.plot(wilson_coeffs, sigmas, "o-", lw=2)
plt.axvline(0, color="gray", ls="--", alpha=0.5, label="SM (c_WWW = 0)")
plt.xlabel(r"$c_{WWW}/\Lambda^2$  [TeV$^{-2}$]")
plt.ylabel(r"$\sigma(e^+e^-\to W^+W^-)$  [pb]")
plt.title(rf"SMEFT $\mathcal{{O}}_{{WWW}}$ at $\sqrt{{s}} = {sqrts/1000:.0f}$ TeV (toy SM piece)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Example C — Leptophobic Z' (custom couplings)

Many DM models predict a Z' that couples preferentially to quarks. Add the q q̄ → Z' → χχ̄ amplitude (analogous to the bundled e+e- → χχ̄ but with quark couplings).

In [ ]:
g_Zq, g_Zchi, M_Zp, Gamma_Zp = sp.symbols("g_Zq g_Zchi M_Zp Gamma_Zp", positive=True)

# s-channel Z' Breit-Wigner; massless quarks, scalar DM
msq_Zp = (g_Zq**2 * g_Zchi**2 / 12) * (s - 4*m_chi**2) * s / ((s - M_Zp**2)**2 + M_Zp**2 * Gamma_Zp**2)

for q in ["u", "d", "c", "s", "b"]:
    register_curated_amplitude(
        process=f"{q} {q}~ -> chi chi~",
        theory="BSM",
        msq=msq_Zp,
        description=f"Leptophobic Z' s-channel: {q}{q}̄ → Z'* → χχ̄",
        notes="Universal q-coupling g_Zq across all 5 active flavours; could be flavour-dependent in real models.",
        overwrite=True,
    )
print("Registered leptophobic Z' for all 5 quark initial states.")

In [ ]:
# Resonance scan
energies = np.linspace(50, 500, 40)
sigmas = []
for e in energies:
    r = total_cross_section(
        "u u~ -> chi chi~", "BSM", sqrt_s=float(e),
        coupling_vals={"g_Zq": 0.1, "g_Zchi": 1.0,
                       "M_Zp": 250.0, "Gamma_Zp": 5.0, "m_chi": 10.0},
    )
    sigmas.append(r["sigma_pb"])

plt.figure(figsize=(8, 4.5))
plt.plot(energies, sigmas, lw=2, color="darkred")
plt.axvline(250, color="gray", ls="--", label=r"$M_{Z'} = 250$ GeV")
plt.xlabel(r"$\sqrt{\hat s}$ [GeV]")
plt.ylabel(r"$\hat\sigma(u\bar u \to \chi\bar\chi)$ [pb]")
plt.title("Leptophobic Z' resonance — partonic σ̂")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Differential observables for your registered amplitude

Once registered, **everything just works** — including the histogrammed dσ/dX endpoint.

In [ ]:
from feynman_engine.amplitudes.differential import differential_distribution

edges = np.linspace(-0.95, 0.95, 21)
hist = differential_distribution(
    "u u~ -> chi chi~", "BSM", sqrt_s=300.0,
    observable="cos_theta", bin_edges=edges,
    coupling_vals={"g_Zq": 0.1, "g_Zchi": 1.0,
                   "M_Zp": 250.0, "Gamma_Zp": 5.0, "m_chi": 10.0},
)
centers = np.array(hist["bin_centers"])
ds = np.array(hist["dsigma_dX_pb"])

plt.figure(figsize=(7, 4))
plt.bar(centers, ds, width=0.085, color="crimson", edgecolor="darkred")
plt.xlabel(r"$\cos\theta^*$")
plt.ylabel(r"$d\sigma/d\cos\theta$ [pb]")
plt.title(r"Leptophobic Z' DM production: $u\bar u \to \chi\bar\chi$ near resonance")
plt.tight_layout(); plt.show()

## What's next for BSM work

- **Add the model to the QGRAF model file** (`contrib/qgraf/models/bsm.mod`) if you want diagram enumeration too — otherwise registration alone gives you the symbolic |M̄|² and everything that follows from it.
- **Hadronic σ for your BSM process**: register the partonic |M̄|² and call `hadronic_cross_section("p p -> ...")` — the engine convolves with PDFs automatically.
- **Multiple operators**: register one process per BSM operator, then sum or compare in your analysis script.
- **Custom theory tag**: pass `theory="MyModel"` and the engine treats it as a separate registry; you can mix-and-match with built-in `EW` / `BSM` / `QCD` formulas as needed.
- **Frontend**: any registered amplitude shows up at `/api/amplitude/processes` and is queryable from the Browser UI.